### Config

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
from hydra import compose, initialize
from omegaconf import OmegaConf

initialize(config_path="../config", version_base="1.3")
cfg = compose(config_name="config")
print(OmegaConf.to_yaml(cfg))

sys.path.insert(0, str(cfg.paths.project_root))

paths:
  project_root: /home/p84400019/projects/consciousness-llms/IT-LLMs/
  model_path: ${model.company}/${model.model_family}/${model.model_size}/${model.it}/
  activation_method: ${time_series.node_type}/${time_series.node_activation}/${time_series.projection_method}/
  phyid_method: ${paths.activation_method}phyid_tau-${phyid.tau}/phyid_kind-${phyid.kind}/phyid_redundancy-${phyid.redundancy}/
  data_dir: ${paths.project_root}data/${paths.model_path}${generation.name}/
  data_activations_dir: ${paths.data_dir}activations/
  data_activations_file: ${paths.data_activations_dir}multi_prompt_activations.pkl
  data_phyid_dir: ${paths.data_dir}phyid/${paths.phyid_method}
  data_phyid_file: ${paths.data_phyid_dir}multi_prompt_phyid.pkl
  data_phyid_file_data_array: ${paths.data_phyid_dir}multi_prompt_phyid.nc
  plot_dir: ${paths.project_root}plots/${paths.model_path}${generation.name}/
  plot_time_series_dir: ${paths.plot_dir}time_series/${paths.activation_method}
  plot_phyid_dir: ${path

### Loading the model

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig, AutoConfig
from src.utils import perturb_model, randomize_model_weights

load_model = True

model_name = cfg.model.hf_name
tokenizer = AutoTokenizer.from_pretrained(
    model_name, 
    trust_remote_code=True
)
if load_model:
    model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        device_map='auto', 
        attn_implementation='eager',  
        trust_remote_code=True
    )
    if cfg.model.it == 'random':
        # randomize_model_weights(model, mean=0.0, std=0.02)
        perturb_model(model, scale=10.0)
    model.generation_config = GenerationConfig.from_pretrained(model_name)
    model.generation_config.pad_token_id = model.generation_config.eos_token_id
    model.eval()

Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.57s/it]


### Record activations, save them, and verify them

In [4]:
from src.activation_recorder import ActivationRecorder, MultiPromptActivations

load_from_disk = False
data_activations_file = cfg.paths.data_activations_file
max_new_tokens=cfg.generation.max_new_tokens
prompts = cfg.generation.prompts
# if it not a list but a list of lists, flatten it
if isinstance(prompts, list) and all(isinstance(p, list) for p in prompts):
    prompts = [item for sublist in prompts for item in sublist]

prompt_template = cfg.model.apply_chat_template
print(f"Using prompt_template: {prompt_template}")

if not load_from_disk:
    recorder = ActivationRecorder(model, tokenizer)
    activations = recorder.record_prompts(prompts, max_new_tokens=max_new_tokens, prompt_template=prompt_template)
    activations.verify_recorded_activations(prompts=prompts, max_new_tokens=max_new_tokens, tokenizer=tokenizer, diff_q_size=True)
    activations.save(data_activations_file)

Using prompt_template: base
Recorder initialized for model: ModelInformation(model_name='meta-llama/Llama-3.2-3B', model_architecture='LlamaForCausalLM', num_layers=28, num_attention_heads_per_layer=24, total_num_attention_heads=672, hidden_size=3072, head_dim=128, n_routed_experts=None, n_shared_experts=None, num_experts_per_tok=None, attention_implementation='default')
Working on prompt 0: Question: Imagine a future where humans have evolved to live underwater. Describe the adaptations they might develop.
 Answer: 

Query states shape: torch.Size([1, 24, 24, 128])
Attention activations for layer_idx in layer 0: 0
Attention activations for queries in layer 0: torch.Size([1, 24, 24, 128])
Attention activations for attention_weights in layer 0: torch.Size([1, 24, 24, 24])
Attention activations for attention_outputs in layer 0: torch.Size([1, 24, 24, 128])
Attention activations for projected_outputs in layer 0: torch.Size([1, 24, 24, 3072])
Query states shape: torch.Size([1, 24, 24, 128]

In [ ]:
# Load the activations from disk.
loaded_activations = MultiPromptActivations.load(data_activations_file)

# Optional: verify the loaded activations match the saved ones.
print("Loaded MultiPromptActivations object has:", len(loaded_activations.prompts), "prompts recorded.")

# Check again the activations
loaded_activations.verify_recorded_activations(prompts=prompts, max_new_tokens=max_new_tokens, tokenizer=tokenizer, diff_q_size=True)

print("Final MultiPromptActivations object has:", len(loaded_activations.prompts), "prompts recorded.")

# Extract the first prompt, first step, first layer, first head
prompt_acts = loaded_activations.prompts[0]
step_acts = prompt_acts.steps[0]
layer_acts = step_acts.layers[0]
attn = layer_acts.attention
for head_idx, head_acts in attn.heads.items():
    print(f"Head {head_idx} activations:")
    print(head_acts.query.shape)
    print(head_acts.attention_weights.shape)
    print(head_acts.attention_outputs.shape)
    print(head_acts.projected_outputs.shape)

MultiPromptActivations successfully loaded from '/home/p84400019/projects/consciousness-llms/IT-LLMs/data/alibaba/qwen-3/0.6B/random/base_prompt/activations/multi_prompt_activations.pkl'.
Loaded MultiPromptActivations object has: 1 prompts recorded.
Activations check passed!
Final MultiPromptActivations object has: 1 prompts recorded.
Head 0 activations:
torch.Size([128])
torch.Size([18])
torch.Size([128])
torch.Size([1024])
Head 1 activations:
torch.Size([128])
torch.Size([18])
torch.Size([128])
torch.Size([1024])
Head 2 activations:
torch.Size([128])
torch.Size([18])
torch.Size([128])
torch.Size([1024])
Head 3 activations:
torch.Size([128])
torch.Size([18])
torch.Size([128])
torch.Size([1024])
Head 4 activations:
torch.Size([128])
torch.Size([18])
torch.Size([128])
torch.Size([1024])
Head 5 activations:
torch.Size([128])
torch.Size([18])
torch.Size([128])
torch.Size([1024])
Head 6 activations:
torch.Size([128])
torch.Size([18])
torch.Size([128])
torch.Size([1024])
Head 7 activations:

In [ ]:
step_acts = prompt_acts.steps[4]
for layer_idx, layer_acts in step_acts.layers.items():
    print(f"Layer {layer_idx} activations:")
    moe_layer_acts = step_acts.layers[2].moe
    for expert_id, expert_acts in moe_layer_acts.experts.items():
        print(f"Expert {expert_id} activations:")
        print(repr(expert_acts))
        print("Gate value:", expert_acts.gate_value)
        print("MLP output:", expert_acts.mlp_output)
        print("Expert output:", expert_acts.expert_output)

Layer 0 activations:


AttributeError: 'NoneType' object has no attribute 'experts'